# Protein Prep

This notebook demonstrates how to use Protein Prep functionality on Deep Origin

We go over:

1. Structure Report
2. Preparing 1eby -- pocket from crystal ligand
3. Preparing 5qsp -- loop modelling 


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import BRD_DATA_DIR, Protein, ProteinPrep, StructureReport, Ligand
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
client

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.id

## 1. Structure Report

In this section we demonstrate how we can generate a structure report on Protein inputs

### Structure Report from a file

Here, we run a structure report on a file

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
sr = StructureReport(protein=protein)
results = sr.run()

In [ ]:
results[0]

### Structure report from a PDB ID

We can also run a structure report for any entry in RCSB using a PDB ID. This structure report is generated without downloading the file, using metadata from RCSB

In [ ]:
sr = StructureReport(pdb_id="6GOG")
results = sr.run()
results[0]

In [ ]:
sr = StructureReport(pdb_id="1EW3")
results = sr.run()
results[0]

## 2 Preparing 1eby 

In this section we walk through preparing `1eby`. Here, we do the following:

- get reccomendations on what to keep and what to remove
- act on those reccomendations
- remove crystal ligand
- save crystal ligand as a pose 

In [ ]:
protein = Protein.from_pdb_id("1eby")
protein.sync()
protein.show()

In [ ]:
prep = ProteinPrep(protein=protein, client=client)
df = prep.recommend()
df

We can inspect the non-water components and their reccomendations: 

In [ ]:
df[df["kind"] != "water"]

Now we can apply transformations using method chaining:

In [ ]:
(
    prep
    .keep(kind="water", subtype="coordinating")
    .skip(kind="water", subtype="crystal")
    .skip(decision="review")
)


## Run Protein prep

Now we have made our decision, we can run protein prep. 

In [ ]:
prep.model_missing_loops = False
prepared = prep.run()


In [ ]:
prepared.download()
prepared.show()

In [ ]:
prepared.id

In [ ]:
poses = prep.get_crystal_poses()
pose = poses[0]
pose.download()
pose.show()

In [ ]:
from itertools import islice

with open(prepared.local_path) as f:
    print("".join(islice(f, 15)), end="")

## Loop Modelling

In this section, we demonstrate how we can also model loops 

In [ ]:
protein = Protein.from_pdb_id("5qsp")
protein.show()

In [ ]:
sr = StructureReport(protein=protein)
results = sr.run()

In [ ]:
results[0]

In [ ]:
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()

In [ ]:
(
    prep
    .skip(decision="review")
)

In [ ]:
prep.start()

In [ ]:
await prep.watch()

In [ ]:
prep.get_user_logs()

In [ ]:
prepped_protein = prep.get_results()

In [ ]:
prepped_protein.download()
prepped_protein.show()

In [ ]:
prepped_protein.local_path

## Issues to fix


3. need to emit warnings if we don't use alphafold, etc.
4. if i send in 1eby, crystal ligand should be extracted, SDF saved, ligand entry in data platform ligand table. 



## Prepare Protein and find novel pockets

In this example we prepare a protein and find pockets in a single exeuction

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()

In [ ]:
prep = ProteinPrep(protein=protein, client=client)
df = prep.recommend()
df

## Prepare Protein and make a pocket from crystal ligand

In this example, we use a holo structure and do the following things:

1. Extract the crystal ligand pose
2. Register Pose with data platform
3. Create a pocket from the crystal ligand structure